## 整体思路-方案一(推荐起步)：3D 圆弧重建 + 3D 中点
这是最干净、误差最可控的方案。流程如下。
#### 第 1 步：左右目分别得到 mask(你已有)。
立体校正左右图，使极线水平。
#### 第 2 步：mask → 骨架点集。
对每个 mask 做形态学骨架化(skimage.morphology.skeletonize),得到一串中线像素 {(u, v)}。骨架已经是亚像素之前最干净的"针体中线"表示。
### 第 3 步：沿极线匹配左右骨架点。
校正后,左目骨架上每点 (uL, vL) 的对应点在右目同一行 v=vL 上。在右目骨架点集里找 v 最接近 vL 的点 (uR, vL'),得到一对匹配。注意:
针可能是近似垂直方向,这时同一个 v 对应多个 u(骨架自交或弧线两端 v 相同)。处理办法:按弧长参数化,而不是按 v 索引。具体见下一步。
#### 第 4 步：按弧长参数化骨架,统一参数范围。
- 对左目骨架点按相邻距离累加,得到弧长 sL ∈ [0, LL],归一化为 tL ∈ [0, 1]。
- 右目同样得到 tR ∈ [0, 1]。
- 然而 tL=0.5 和 tR=0.5 仍然不是同一物理点(可见弧段范围不同)。所以这一步只是为了得到有序的骨架点。
#### 第 5 步：逐点三角化。
对左目骨架上每个点,在右目骨架上沿极线(同一 v 行)找最近的点,用 cv2.triangulatePoints 三角化,得到一系列 3D 点 {Pi}。这些点应当近似落在同一个 3D 圆上。
#### 第 6 步：3D 圆拟合。针是平面圆弧,所以 {Pi} 共面共圆。
- a. 拟合平面:对 {Pi} 做 PCA,最小特征值方向是平面法向量 n,平面过点云质心 c。
- b. 投影到平面 2D 坐标系:在该平面内建立局部 2D 坐标 (e1, e2)(PCA 的两个主方向),把 {Pi} 投影成 2D 点 {pi}。
- c. 2D 圆拟合:对 {pi} 做代数最小二乘圆拟合(你代码里 _fit_circle_lsq 直接复用),得到 2D 圆心 c2D 和半径 R。
- d. 回到 3D:c2D 变换回 3D 得到 3D 圆心 C,半径 R,法向量 n。
#### 第 7 步：在 3D 圆上取中点。
- 把每个 Pi 投到 3D 圆上,计算其极角 θi(在平面局部 2D 坐标里 atan2)。
- 排序 θi,找最大角度间隙(对应针的开口),弧的两端点 θstart、θend 即间隙两侧。
- 中点角度 θmid = (θstart + θend) / 2。
- 3D 中点: M = C + R·(cos(θmid)·e1 + sin(θmid)·e2)。

这个 M 就是机器人要的夹取点,在相机坐标系下。
优点:
中点是真正的 3D 物理量,不依赖于左右 2D 中点的对应。
圆拟合对骨架噪声、断裂、部分遮挡天然鲁棒——只要左右目加起来覆盖了足够的弧段,3D 圆就能定下来。
副产品 R 和 n 还能告诉机器人针的姿态(针所在平面的朝向),夹爪可以选择垂直于针平面进入,这对实际抓取非常有用。

注意点:
极线匹配时如果右目同一 v 行没有骨架点,跳过该点。
三角化前先用 cv2.stereoRectify + cv2.initUndistortRectifyMap 做校正,或者直接用未校正的 cv2.triangulatePoints 加左右投影矩阵 P1、P2(后者更通用,不需要重映射图像)。
可以加个 RANSAC 包在 3D 圆拟合外面,剔除明显偏离的三角化外点(通常来自反光区域的骨架毛刺)。
“物理上：手术针是圆弧
工业生产的缝合针（1/4 圆、3/8 圆、1/2 圆、5/8 圆等规格）在制造层面就是精确的平面圆弧——它们由直针绕一个固定半径的芯轴弯制而成。3/8 圆针（最常见）就是真实的、平面的、半径恒定的 3/8 圆周。”

## 整体流程

导入 + 加载占位标定参数(注释里给了真实标定的加载函数)。
复用你的 run_pipeline 跑左右图的 YOLO+SAM3,得到两个 mask。
骨架提取:keep_large_components 保留所有合理大小的连通域(对反光断裂鲁棒),order_skeleton 用图遍历把骨架像素按弧长方向排好序。
cv2.undistortPoints 去畸变(D 全零时是空操作)。
极线匹配:从 K、R、T 直接构造基础矩阵 F,逐点找右图离极线最近的骨架点,丢弃距离 > 3 px 的。
cv2.triangulatePoints 三角化。
3D 圆 RANSAC 拟合:PCA 求平面 → 平面内 2D 圆代数拟合 → 变回 3D,RANSAC 阈值 0.5 mm。
在 3D 圆所在平面里复用你 fit_needle_arc 的"最大角度间隙找端点 → 中点是端点角度均值"逻辑,得到 3D 中点。
把 3D 中点反投影回左右图做视觉验证。
3D 可视化 + 保存 .npz(midpoint、plane normal、radius、RMS、inlier 数)。

几个使用要点

单位:所有 3D 输出跟 T 的单位一致。我默认按 mm 设置(基线 5 mm),你换成真实标定时要保证 T 也是 mm。
占位标定:第 1 节里的 K1/K2/R/T 是合理的内窥镜估值,可以让 notebook 跑通,但真实精度需要替换为 cv2.stereoCalibrate 的输出。代码里留了 load_calibration_npz 函数。
平面法向量副产品:out['plane_normal'] 是针所在平面的 3D 法向量,机器人夹爪可以选择垂直于这个法向量进入,这是手术针自动抓取的标准朝向。
RMS 残差是质量门:第 8 节末尾给了诊断阈值。机器人系统里建议加一个守门:RMS > 阈值就丢弃这一帧,等下一帧。

# Stereo 3D Needle Midpoint Reconstruction

**Pipeline**
1. Run YOLO + SAM3 on left and right images (reusing `detection_to_segmentation.py`).
2. Extract 2D skeleton from each mask.
3. Match skeleton points across views using **epipolar geometry** (uncalibrated / un-rectified images).
4. **Triangulate** matched 2D pairs into a 3D point cloud.
5. Fit a **3D circle** (PCA plane + 2D circle fit + transform back).
6. Recover the **arc midpoint** as the gripping target for the robot.

**Inputs**: left image path, right image path, stereo calibration (K1, K2, D1, D2, R, T).

**Output**: 3D midpoint coordinates in the **left camera frame**, plus needle plane normal and radius for grasp planning.

## 0. Imports and config

In [ ]:
import sys
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.morphology import skeletonize
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# Make detection_to_segmentation.py importable.
# Adjust this path if your script lives somewhere else.
sys.path.insert(0, '.')
from detection_to_segmentation import (
    load_yolo,
    load_sam3,
    run_pipeline,
)

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline

## 1. Stereo calibration parameters

Replace the placeholder values below with your real calibration once you have it.

Convention used here:
- `K1`, `K2`: 3x3 intrinsics of left and right camera (pixels).
- `D1`, `D2`: distortion coefficients (k1, k2, p1, p2, k3). All zeros if undistorted.
- `R`, `T`: rotation and translation **of right camera relative to left** (the standard OpenCV `stereoCalibrate` output). The left camera frame is the world frame for our 3D output.
- Image size in pixels.

In [ ]:
# ---- Stereo endoscope calibration (from endoscope_calibration.yaml) ----
IMG_W, IMG_H = 800, 600

# Left camera intrinsics (M1)
K1 = np.array([[726.412892, -0.049458, 375.577636],
               [  0.000000, 726.410167, 286.483688],
               [  0.000000,   0.000000,   1.000000]], dtype=np.float64)

# Right camera intrinsics (M2)
K2 = np.array([[732.769622, -0.437122, 448.092014],
               [  0.000000, 732.293867, 286.816235],
               [  0.000000,   0.000000,   1.000000]], dtype=np.float64)

# Distortion coefficients [k1, k2, p1, p2, k3]
D1 = np.array([-0.037295,  0.074764, -0.009469,  0.003614, 0.0], dtype=np.float64)
D2 = np.array([-0.034440,  0.184112, -0.006644,  0.010357, 0.0], dtype=np.float64)

# Rotation of right camera relative to left (R)
R = np.array([[ 0.999918,  0.003107, -0.012429],
              [-0.003106,  0.999995,  0.000162],
              [ 0.012430, -0.000123,  0.999923]], dtype=np.float64)

# Translation of right camera relative to left (T), unit: mm
# Note: baseline in YAML is 0.004534 m = 4.534 mm, consistent with T[0] below.
T = np.array([[-4.527460],
              [-0.047894],
              [-0.242786]], dtype=np.float64)

# Helper: load from a .npz if you have one.
def load_calibration_npz(path):
    """Expected keys: K1, K2, D1, D2, R, T, img_w, img_h."""
    d = np.load(path)
    return (d['K1'], d['K2'], d['D1'], d['D2'],
            d['R'], d['T'].reshape(3, 1),
            int(d['img_w']), int(d['img_h']))

# K1, K2, D1, D2, R, T, IMG_W, IMG_H = load_calibration_npz('stereo_calib.npz')

print('K1 =\n', K1)
print('K2 =\n', K2)
print('R  =\n', R)
print('T  =\n', T.ravel(), '  (units: mm)')

## 2. Left and right image paths, and YOLO weights

In [ ]:
LEFT_IMG  = '/home/songyu/Datasets/Autosurg/cuhk/Cali_Data_Needle_Image/needle_image/left/img_0001.jpg'    # <-- set me (left image)
RIGHT_IMG = '/home/songyu/Datasets/Autosurg/cuhk/Cali_Data_Needle_Image/needle_image/right/img_0001.jpg'   # <-- set me (right image)

YOLO_WEIGHTS = '/home/songyu/Project/yolo/YOLOv8_needle/runs/detect/runs/needle_finetune/cuhk_v1/weights/best.pt'  # <-- set me
YOLO_CONF    = 0.45
BOX_EXPAND   = 1.1

OUTPUT_DIR = Path('outputs_3d'); OUTPUT_DIR.mkdir(exist_ok=True)

## 3. Run YOLO + SAM3 on both views

Models are loaded **once** and reused for both images. We disable the 2D arc fitting in `run_pipeline` because the real arc fitting now happens in 3D.

In [ ]:
yolo_model = load_yolo(YOLO_WEIGHTS)
sam3_model, sam3_processor = load_sam3()

left_result = run_pipeline(
    LEFT_IMG, YOLO_WEIGHTS,
    conf=YOLO_CONF, expand=BOX_EXPAND, fit_arc=False,
    output_path=str(OUTPUT_DIR),
    yolo_model=yolo_model,
    sam3_model=sam3_model, sam3_processor=sam3_processor)

right_result = run_pipeline(
    RIGHT_IMG, YOLO_WEIGHTS,
    conf=YOLO_CONF, expand=BOX_EXPAND, fit_arc=False,
    output_path=str(OUTPUT_DIR),
    yolo_model=yolo_model,
    sam3_model=sam3_model, sam3_processor=sam3_processor)

assert left_result is not None and right_result is not None, \
    'YOLO failed to detect the needle in one of the views'

mask_L = left_result['masks'][0]    # bool, shape (H, W)
mask_R = right_result['masks'][0]
print('mask_L shape:', mask_L.shape, '  pixels:', int(mask_L.sum()))
print('mask_R shape:', mask_R.shape, '  pixels:', int(mask_R.sum()))

## 4. Skeleton extraction

Skeletonization reduces the mask to a 1-pixel-wide centerline — the 2D projection of the needle's 3D centerline. We:
1. Keep all reasonably large connected components (a specular reflection often splits the mask).
2. Skeletonize.
3. **Order** the skeleton pixels along the curve. Ordering matters because we'll match left/right points in matching order along the arc, and the unordered set returned by `np.where` is row-major garbage.

We order skeleton pixels by performing a depth-first walk from one endpoint to the other along 8-connected neighbors. Endpoints are skeleton pixels with exactly one neighbor.

In [ ]:
def keep_large_components(mask, min_ratio=0.15):
    """Keep all CCs whose area >= min_ratio * largest CC area.
    Robust to a needle mask split into 2-3 fragments by reflections."""
    m = mask.astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n <= 1:
        return mask.astype(bool)
    areas = stats[1:, cv2.CC_STAT_AREA]
    if len(areas) == 0:
        return mask.astype(bool)
    threshold = areas.max() * min_ratio
    keep = np.zeros_like(m, dtype=bool)
    for i, a in enumerate(areas, start=1):
        if a >= threshold:
            keep |= (labels == i)
    return keep


def order_skeleton(skel):
    """Order skeleton pixels along the curve.
    Returns a (N, 2) array of (u, v) = (x, y) pixel coordinates in walk order.
    For broken skeletons, returns the longest chain."""
    skel = skel.astype(np.uint8)
    H, W = skel.shape
    pix = set(map(tuple, np.argwhere(skel)))  # (y, x) pairs
    if not pix:
        return np.empty((0, 2), dtype=np.float32)

    nbrs8 = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]

    def neighbors(p):
        y, x = p
        out = []
        for dy, dx in nbrs8:
            q = (y+dy, x+dx)
            if q in pix:
                out.append(q)
        return out

    # Find endpoints (degree-1 pixels). If none (closed loop), pick any.
    endpoints = [p for p in pix if len(neighbors(p)) == 1]
    start = endpoints[0] if endpoints else next(iter(pix))

    # Greedy walk: at each step move to an unvisited neighbor; if multiple,
    # pick the one with the smallest in-degree (favors chain continuation).
    chain = [start]
    visited = {start}
    while True:
        cur = chain[-1]
        cands = [q for q in neighbors(cur) if q not in visited]
        if not cands:
            break
        # Prefer 4-connected neighbors first to avoid skipping diagonally.
        cands.sort(key=lambda q: (abs(q[0]-cur[0]) + abs(q[1]-cur[1]), q))
        nxt = cands[0]
        chain.append(nxt)
        visited.add(nxt)

    # Convert (y, x) -> (x, y) = (u, v)
    arr = np.array([(x, y) for (y, x) in chain], dtype=np.float32)
    return arr


def extract_skeleton(mask):
    cleaned = keep_large_components(mask, min_ratio=0.15)
    skel = skeletonize(cleaned)
    ordered = order_skeleton(skel)
    return ordered, cleaned, skel


skel_L, clean_L, raw_skel_L = extract_skeleton(mask_L)
skel_R, clean_R, raw_skel_R = extract_skeleton(mask_R)
print(f'left  skeleton: {len(skel_L):4d} ordered pixels')
print(f'right skeleton: {len(skel_R):4d} ordered pixels')

In [ ]:
# Visualize skeletons over the original images.
img_L = cv2.cvtColor(cv2.imread(LEFT_IMG),  cv2.COLOR_BGR2RGB)
img_R = cv2.cvtColor(cv2.imread(RIGHT_IMG), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, img, skel, title in zip(axes,
                                 [img_L, img_R],
                                 [skel_L, skel_R],
                                 ['Left view', 'Right view']):
    ax.imshow(img)
    if len(skel) > 0:
        ax.scatter(skel[:, 0], skel[:, 1], c=np.arange(len(skel)),
                   cmap='viridis', s=2)
    ax.set_title(f'{title} skeleton (color = order along curve)')
    ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Undistort skeleton points

Triangulation expects points that obey the pinhole model. We apply `cv2.undistortPoints` to both skeletons (still in pixel coordinates afterward, but with lens distortion removed). If your `D1`/`D2` are zero this is a no-op.

In [ ]:
def undistort_pixels(pts, K, D):
    """Remove lens distortion. Input/output are (N, 2) pixel coordinates."""
    if len(pts) == 0:
        return pts
    pts_in = pts.reshape(-1, 1, 2).astype(np.float64)
    # P=K means: produce undistorted points back in pixel space (still ideal pinhole).
    pts_out = cv2.undistortPoints(pts_in, K, D, P=K)
    return pts_out.reshape(-1, 2).astype(np.float32)

skel_L_ud = undistort_pixels(skel_L, K1, D1)
skel_R_ud = undistort_pixels(skel_R, K2, D2)

## 6. Epipolar matching of skeleton points

Given an undistorted point in the left image, its corresponding point in the right image must lie on a specific line — the **epipolar line**. We compute the fundamental matrix `F` from `K1`, `K2`, `R`, `T`, then for each left skeleton point we find the right skeleton point closest to its epipolar line.

Two safeguards:
- Reject matches whose perpendicular distance to the epipolar line exceeds a threshold (a few pixels).
- Reject matches with negative depth after triangulation.

**Why F from R, T:** the essential matrix is `E = [t]_x R`; `F = K2^{-T} E K1^{-1}` maps a left pixel to its epipolar line in the right image.

In [ ]:
def skew(v):
    v = v.ravel()
    return np.array([[    0, -v[2],  v[1]],
                     [ v[2],     0, -v[0]],
                     [-v[1],  v[0],     0]], dtype=np.float64)

def fundamental_from_KRT(K1, K2, R, T):
    E = skew(T) @ R
    F = np.linalg.inv(K2).T @ E @ np.linalg.inv(K1)
    return F

F = fundamental_from_KRT(K1, K2, R, T)
print('F =\n', F)

In [ ]:
def epipolar_match(pts_L, pts_R, F, max_dist_px=3.0):
    """For each left point, find the right point closest to its epipolar line.
    Returns (matched_L, matched_R) arrays of equal length, both (M, 2).

    `max_dist_px` is the upper bound on perpendicular distance; matches above
    this threshold are dropped. 2-3 px is appropriate for a well-calibrated rig;
    raise it if your calibration is loose or you see most matches dropped."""
    if len(pts_L) == 0 or len(pts_R) == 0:
        return np.empty((0, 2)), np.empty((0, 2))

    # Epipolar lines in the right image, one per left point: l = F @ [u, v, 1]^T
    ones = np.ones((len(pts_L), 1), dtype=np.float64)
    pL_h = np.hstack([pts_L.astype(np.float64), ones])      # (N, 3)
    lines = (F @ pL_h.T).T                                  # (N, 3): a, b, c
    # Normalize so that sqrt(a^2 + b^2) = 1 -> distance is |a*u + b*v + c|
    norms = np.linalg.norm(lines[:, :2], axis=1, keepdims=True)
    norms[norms < 1e-12] = 1.0
    lines = lines / norms

    pR = pts_R.astype(np.float64)                           # (M, 2)
    # Distances: (N, M) = |a_i*u_j + b_i*v_j + c_i|
    dists = np.abs(lines[:, [0]] * pR[:, 0][None, :] +
                   lines[:, [1]] * pR[:, 1][None, :] +
                   lines[:, [2]])

    best_idx = dists.argmin(axis=1)
    best_d   = dists[np.arange(len(pts_L)), best_idx]
    keep = best_d <= max_dist_px

    return pts_L[keep], pts_R[best_idx[keep]]

matches_L, matches_R = epipolar_match(skel_L_ud, skel_R_ud, F, max_dist_px=3.0)
print(f'kept {len(matches_L)} / {len(skel_L_ud)} left points after epipolar matching')

In [ ]:
# Quick sanity plot: a few epipolar lines and their matched right-image points.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_L); axes[0].set_title('Left'); axes[0].axis('off')
axes[1].imshow(img_R); axes[1].set_title('Right + epipolar lines'); axes[1].axis('off')

n_show = min(8, len(matches_L))
if n_show > 0:
    show_idx = np.linspace(0, len(matches_L) - 1, n_show).astype(int)
    colors = plt.cm.tab10(np.arange(n_show) % 10)
    for k, idx in enumerate(show_idx):
        uL, vL = matches_L[idx]
        uR, vR = matches_R[idx]
        axes[0].scatter([uL], [vL], c=[colors[k]], s=40, edgecolors='white')
        # Epipolar line in right image: a*u + b*v + c = 0
        a, b, c = F @ np.array([uL, vL, 1.0])
        x = np.array([0, IMG_W])
        if abs(b) > 1e-6:
            y = -(a * x + c) / b
            axes[1].plot(x, y, color=colors[k], linewidth=1)
        axes[1].scatter([uR], [vR], c=[colors[k]], s=40, edgecolors='white')
plt.tight_layout(); plt.show()

## 7. Triangulation

Build the projection matrices and triangulate every matched pair. The left camera frame is the world frame, so:

$$P_1 = K_1 [I \,|\, 0], \qquad P_2 = K_2 [R \,|\, T].$$

Output 3D points are in **mm** (matching the units of `T`).

In [ ]:
def projection_matrices(K1, K2, R, T):
    P1 = K1 @ np.hstack([np.eye(3), np.zeros((3, 1))])
    P2 = K2 @ np.hstack([R, T])
    return P1, P2

def triangulate(P1, P2, pts1, pts2):
    """pts1, pts2: (N, 2). Returns (N, 3) in the left camera frame."""
    if len(pts1) == 0:
        return np.empty((0, 3))
    pts1_h = pts1.T.astype(np.float64)  # (2, N)
    pts2_h = pts2.T.astype(np.float64)
    X4 = cv2.triangulatePoints(P1, P2, pts1_h, pts2_h)  # (4, N)
    X3 = (X4[:3] / X4[3]).T
    return X3

P1, P2 = projection_matrices(K1, K2, R, T)
pts3d  = triangulate(P1, P2, matches_L, matches_R)

# Drop points behind the camera or absurdly far.
valid = (pts3d[:, 2] > 0) & (pts3d[:, 2] < 1000.0)  # mm
pts3d = pts3d[valid]
print(f'triangulated {len(pts3d)} valid 3D points')
print(f'  Z range:   {pts3d[:, 2].min():.2f} .. {pts3d[:, 2].max():.2f} mm')
print(f'  centroid:  {pts3d.mean(axis=0)}')

## 8. 3D circle fit

**Algorithm**
1. Centroid `c = mean(pts3d)`.
2. PCA on `pts3d - c`. The two largest principal vectors `e1, e2` span the best-fit plane; the smallest is the plane normal `n`.
3. Project all points to plane coordinates `(p, q) = ((P - c) . e1, (P - c) . e2)`.
4. Algebraic least-squares circle fit on the 2D `(p, q)` set: `p^2 + q^2 + D p + E q + F = 0` -> center `(-D/2, -E/2)`, radius `sqrt(p_c^2 + q_c^2 - F)`.
5. Map the 2D circle center back to 3D: `C_3D = c + p_c * e1 + q_c * e2`.
6. Optional RANSAC wrapper to reject outliers — useful if a few specular pixels gave bad matches.

In [ ]:
def fit_plane_pca(pts):
    """Returns centroid c (3,), basis e1, e2 (3,), normal n (3,)."""
    c = pts.mean(axis=0)
    centered = pts - c
    # SVD: rows are points; right singular vectors are principal axes.
    _, _, Vt = np.linalg.svd(centered, full_matrices=False)
    e1 = Vt[0]
    e2 = Vt[1]
    n  = Vt[2]
    return c, e1, e2, n

def fit_circle_2d(p, q):
    """Algebraic LSQ. Returns (pc, qc, r) or None."""
    A = np.column_stack([p, q, np.ones_like(p)])
    b = -(p**2 + q**2)
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    D, E, F = sol
    pc, qc = -D / 2.0, -E / 2.0
    r2 = pc**2 + qc**2 - F
    if r2 <= 0:
        return None
    return float(pc), float(qc), float(np.sqrt(r2))

def fit_circle_3d(pts, ransac_iters=200, ransac_thresh=0.5, rng=None):
    """Fit a 3D circle to pts (N, 3) with RANSAC.
    ransac_thresh: max distance (mm) from a 3D point to the fitted circle for it to be an inlier.
    Returns dict with center, radius, normal, e1, e2, inlier mask, and residual."""
    pts = np.asarray(pts, dtype=np.float64)
    n_pts = len(pts)
    if n_pts < 5:
        return None
    rng = rng or np.random.default_rng(0)

    def _fit_once(subset):
        c, e1, e2, n = fit_plane_pca(subset)
        p = (subset - c) @ e1
        q = (subset - c) @ e2
        circ = fit_circle_2d(p, q)
        if circ is None:
            return None
        pc, qc, r = circ
        C3 = c + pc * e1 + qc * e2
        return C3, r, n, e1, e2

    def _residuals(model, all_pts):
        C3, r, n, e1, e2 = model
        # Distance from each point to the 3D circle:
        # 1) signed distance to plane,
        # 2) in-plane distance from projected point to circle.
        d_plane = (all_pts - C3) @ n
        proj    = all_pts - np.outer(d_plane, n)
        d_inplane = np.linalg.norm(proj - C3, axis=1) - r
        return np.sqrt(d_plane**2 + d_inplane**2)

    best_inliers = None
    best_model   = None
    sample_size  = max(5, int(0.1 * n_pts))

    for _ in range(ransac_iters):
        idx = rng.choice(n_pts, size=sample_size, replace=False)
        m = _fit_once(pts[idx])
        if m is None:
            continue
        res = _residuals(m, pts)
        inliers = res < ransac_thresh
        if best_inliers is None or inliers.sum() > best_inliers.sum():
            best_inliers = inliers
            best_model   = m

    if best_model is None:
        return None

    # Refit on all inliers for a final, lower-variance estimate.
    final = _fit_once(pts[best_inliers])
    if final is None:
        final = best_model
    C3, r, n, e1, e2 = final
    res = _residuals(final, pts[best_inliers])
    rms = float(np.sqrt(np.mean(res**2)))

    return {
        'center':  C3,
        'radius':  r,
        'normal':  n / np.linalg.norm(n),
        'e1':      e1,
        'e2':      e2,
        'inliers': best_inliers,
        'rms_mm':  rms,
        'n_in':    int(best_inliers.sum()),
        'n_total': n_pts,
    }

circle = fit_circle_3d(pts3d, ransac_iters=300, ransac_thresh=0.5)
assert circle is not None, '3D circle fit failed'

print(f"center  : {circle['center']}  mm")
print(f"radius  : {circle['radius']:.3f} mm")
print(f"normal  : {circle['normal']}")
print(f"inliers : {circle['n_in']} / {circle['n_total']}")
print(f"RMS res : {circle['rms_mm']:.3f} mm")

### Quality check

- **Radius**: a typical surgical needle is 4–25 mm. If you get something far outside this range, the calibration units or the matches are off.
- **RMS residual**: for an in-focus, well-calibrated rig you should see well under 0.3 mm. Above ~0.5 mm suggests bad matches; above 1 mm suggests bad calibration.
- **Inlier ratio**: > 80% is good; < 50% means you should look at the skeleton/matching steps.

## 9. Find the arc midpoint in 3D

Same idea as your existing `fit_needle_arc`, but operating on the **planar 2D coordinates** of the inlier points (in the fitted plane) instead of pixel coordinates:
1. Compute polar angle of each inlier about the 2D circle center.
2. The largest angular gap is the needle's opening; the two angles bordering it are the arc endpoints.
3. The midpoint angle bisects the arc, **not** the gap.
4. Map back to 3D.

In [ ]:
def arc_midpoint_3d(pts3d, circle):
    inliers = pts3d[circle['inliers']]
    C  = circle['center']
    r  = circle['radius']
    e1 = circle['e1']
    e2 = circle['e2']
    n  = circle['normal']

    # Project inliers to plane 2D coords (around the centroid we used to build the basis).
    centered = inliers - C
    p = centered @ e1
    q = centered @ e2
    angles = np.arctan2(q, p)              # in (-pi, pi]

    sorted_a = np.sort(angles)
    # Wrap-around gap.
    gaps = np.diff(np.concatenate([sorted_a, [sorted_a[0] + 2 * np.pi]]))
    gap_idx = int(np.argmax(gaps))

    a_end   = float(sorted_a[gap_idx])
    a_start = float(sorted_a[(gap_idx + 1) % len(sorted_a)])
    a_end_u = a_end if a_end > a_start else a_end + 2 * np.pi
    a_mid   = 0.5 * (a_start + a_end_u)
    sweep   = a_end_u - a_start

    mid_3d   = C + r * (np.cos(a_mid)   * e1 + np.sin(a_mid)   * e2)
    end1_3d  = C + r * (np.cos(a_start) * e1 + np.sin(a_start) * e2)
    end2_3d  = C + r * (np.cos(a_end)   * e1 + np.sin(a_end)   * e2)

    return {
        'midpoint_3d':  mid_3d,
        'endpoint1_3d': end1_3d,
        'endpoint2_3d': end2_3d,
        'arc_sweep_deg': float(np.degrees(sweep)),
        'mid_angle':    a_mid,
        'plane_normal': n,
    }

arc = arc_midpoint_3d(pts3d, circle)

print('=== RESULT ===')
print(f"3D MIDPOINT (in left camera frame): {arc['midpoint_3d']}  mm")
print(f"  depth Z         : {arc['midpoint_3d'][2]:.3f} mm")
print(f"  arc sweep       : {arc['arc_sweep_deg']:.1f} deg")
print(f"  needle radius   : {circle['radius']:.3f} mm")
print(f"  plane normal    : {arc['plane_normal']}")
print(f"  endpoint 1      : {arc['endpoint1_3d']}")
print(f"  endpoint 2      : {arc['endpoint2_3d']}")

## 10. Reproject the 3D midpoint onto both images (verification)

If the reprojected midpoint lands on the actual needle in **both** views (and visually near the middle of the visible arc), the pipeline is working. If it's off in one view but on in the other, the right-side match was wrong.

In [ ]:
def project(P, X3):
    """Project a 3D point X3 (3,) with a 3x4 matrix P. Returns (u, v)."""
    Xh = np.array([X3[0], X3[1], X3[2], 1.0])
    x  = P @ Xh
    return x[0] / x[2], x[1] / x[2]

uL_mid, vL_mid = project(P1, arc['midpoint_3d'])
uR_mid, vR_mid = project(P2, arc['midpoint_3d'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_L); axes[0].set_title('Left: reprojected midpoint')
axes[1].imshow(img_R); axes[1].set_title('Right: reprojected midpoint')
for ax in axes: ax.axis('off')
if len(skel_L_ud):
    axes[0].scatter(skel_L_ud[:, 0], skel_L_ud[:, 1], s=1, c='cyan', alpha=0.5)
if len(skel_R_ud):
    axes[1].scatter(skel_R_ud[:, 0], skel_R_ud[:, 1], s=1, c='cyan', alpha=0.5)
axes[0].scatter([uL_mid], [vL_mid], s=200, marker='+', c='red', linewidths=3)
axes[1].scatter([uR_mid], [vR_mid], s=200, marker='+', c='red', linewidths=3)
plt.tight_layout(); plt.show()

## 11. 3D visualization

Plot the 3D point cloud, the fitted circle, and the midpoint.

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Inliers and outliers
inl  = circle['inliers']
outl = ~inl
ax.scatter(pts3d[inl, 0],  pts3d[inl, 1],  pts3d[inl, 2],
           c='royalblue', s=8, label=f'inliers (n={int(inl.sum())})')
if outl.any():
    ax.scatter(pts3d[outl, 0], pts3d[outl, 1], pts3d[outl, 2],
               c='lightgray', s=8, label=f'outliers (n={int(outl.sum())})')

# Fitted full circle
C, r, e1, e2 = circle['center'], circle['radius'], circle['e1'], circle['e2']
thetas = np.linspace(0, 2 * np.pi, 200)
circ_pts = C[None, :] + r * (np.outer(np.cos(thetas), e1) + np.outer(np.sin(thetas), e2))
ax.plot(circ_pts[:, 0], circ_pts[:, 1], circ_pts[:, 2],
        c='orange', linewidth=1, alpha=0.7, label=f'fitted circle (r={r:.2f} mm)')

# Midpoint
M = arc['midpoint_3d']
ax.scatter([M[0]], [M[1]], [M[2]], c='red', s=120, marker='*',
           label=f'3D midpoint  Z={M[2]:.2f} mm')

ax.set_xlabel('X (mm)'); ax.set_ylabel('Y (mm)'); ax.set_zlabel('Z (mm)')
ax.set_title('3D needle reconstruction (left camera frame)')
ax.legend(); plt.tight_layout(); plt.show()

## 12. Save the result for downstream use

Save the grasp target plus the needle plane normal — the latter lets the robot's grasp planner orient the gripper jaws **perpendicular to the needle plane**, which is the natural approach direction.

In [ ]:
out = {
    'midpoint_mm':   arc['midpoint_3d'],
    'plane_normal':  arc['plane_normal'],
    'radius_mm':     circle['radius'],
    'arc_sweep_deg': arc['arc_sweep_deg'],
    'rms_mm':        circle['rms_mm'],
    'n_inliers':     circle['n_in'],
    'n_total':       circle['n_total'],
}
np.savez(OUTPUT_DIR / 'needle_3d.npz', **out)
print('saved ->', OUTPUT_DIR / 'needle_3d.npz')
for k, v in out.items():
    print(f'  {k:14s}: {v}')

## Notes for tuning

- **`max_dist_px` in `epipolar_match`**: start at 3 px. If you see most matches dropped, your calibration error is larger than that — raise to 5 or check calibration.
- **`ransac_thresh` in `fit_circle_3d`**: 0.5 mm is conservative. With a clean rig and good masks you may push down to 0.2 mm.
- **Needle radius sanity**: hard-code the expected radius range of your actual needles and warn if the fit falls outside. Same for arc sweep (3/8 needle ~135°, 1/2 needle 180°).
- **Multi-frame averaging**: for moving needles, run this every frame and low-pass the midpoint/normal trajectory before passing to the robot — single-frame noise is the dominant error source after calibration.